In [1]:
import os
import random
import copy
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import preprocessing

from utils import get_train_test_split, get_sample_weights, eval_val_data, eval_te_data, event_seperator, reshape_data, CheckMAE, deterministic_ops

2026-04-08 11:08:46.903844: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-08 11:08:46.958095: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-08 11:08:46.958161: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-08 11:08:46.961044: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-08 11:08:46.972648: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# Sets all seeds and environment variables to ensure reproducible results across different runs
deterministic_ops(21)

In [3]:
# IntellEvent is trained on the velocity of the left and right HEEL, ANKLE, and TOE markers 
# using all dimensions (X, Y, Z) resulting in 18 features.
# X = dPlane of movement
# Y = Medial/Lateral
# Z = Up/Down

last_dim = 18

In [4]:
# Flag for IC or FO; 
# If True - IC model will be trained; If False - FO model will be trained
is_IC = True

In [5]:
# Read the dataset
dataset = pd.read_pickle("../datasets/BL1.pkl")


# Purpose:
     This pipeline is designed for training and evaluating machine learning models (specifically IntellEvent) 
     to detect gait events (Initial Contact (IC) and Foot Off (FO)) across diverse pathologies.
# Note:
    For training and testing IntellEvent much larger datasets were used - this is just a representation of how the pipelines work
    with the benchmark laboratory datasets! Results vary drastically!!

## Features (Columns):
### Trajectory: 
     Description: Original 3D marker coordinates for the lower extremities.
     Channels: LHEE (X,Y,Z), LTOE (X,Y,Z), LANK (X,Y,Z), RHEE (X,Y,Z), RTOE (X,Y,Z), RANK (X,Y,Z).
     Format: Stacked list/array per trial with shape (18 features, length_trial).
     Directions: 
         X: Anteroposterior (Forward/Backward)
         Y: Vertical (Side-to-Side)
         Z: Mediolateral (Up and Down)

### Velocity:
     Description: First derivative of the 'Trajectory' column.
     Purpose: Standardized input features used as the primary input for the IntellEvent pipeline.

### All_Events:
     Description: Gait event labeled as they are published in the original research for each benchmark laboratory.
     Format: Array of shape (length_trial).
     Mapping: 0 = No Event; 1 = Left IC; 2 = Left FO; 3 = Right IC; 4 = Right FO.

### GRF_Events:
     Description: Ground Truth labels derived strictly from Force Plate data.
     Purpose: Used as the primary target (label) for model training and MAE evaluation to ensure high precision.
     Mapping: Follows the same 0-4 mapping as 'All_Events'.

### Freq_point:
     Description: The sampling frequency (Hz) at which the marker trajectories were recorded.

### Freq_analog:
     Description: The sampling frequency (Hz) of the force plate (analog) data.

### Trial:
     Description: A unique String identifier for the specific walking trial or file name.

### DBid:
     Description: A unique String identifier for the subject (Patient ID).
     Importance: Used for stratification to ensure the same subject does not appear in both Train and Test sets.

### Label:
     Description: Integer identifying the underlying pathology of the subject.
     Reference: Refer to the specific research paper for the pathology-to-integer mapping (e.g., OD, ND, CP).


In [6]:
dataset

,Trajectory,All_Events,GRF_Events,Freq_point,Freq_analog,Trial,DBid,Label,Velocity
0,"[[-998.6561889648438, -998.598388671875, -998....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",100,1000,SUBJ1 (1).c3d,SUBJ1,1,"[[1.0961537699344401, 1.0960745017520854, 1.09..."
1,"[[2706.868896484375, 2698.872802734375, 2688.1...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",100,1000,SUBJ1 (2).c3d,SUBJ1,1,"[[0.8802452773127574, 0.8434487226310543, 0.77..."
2,"[[-1857.53466796875, -1856.552734375, -1855.43...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",100,1000,SUBJ2 (1).c3d,SUBJ2,1,"[[1.0616849272830562, 1.0594264338904877, 1.05..."
3,"[[2711.50341796875, 2711.138671875, 2710.72753...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",100,1000,SUBJ2 (2).c3d,SUBJ2,1,"[[1.090301889536058, 1.0895344422273747, 1.087..."
4,"[[-1422.18310546875, -1420.97802734375, -1419....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",100,1000,SUBJ3 (3).c3d,SUBJ3,1,"[[1.0544733521884824, 1.0486052408287736, 1.03..."
...,...,...,...,...,...,...,...,...,...
338,"[[-1347.8856201171875, -1347.8297119140625, -1...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",100,1000,BWA05.c3d,TVC46,2,"[[1.1, 1.0999536027851586, 1.0998525231385397,..."
339,"[[-818.3491821289062, -795.0436401367188, -775...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",100,1000,BWA5.c3d,TVC52,2,"[[0.4524659368447178, 0.5000591928807567, 0.59..."
340,"[[-932.9660034179688, -930.478271484375, -928....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",100,1000,BWA7.c3d,TVC52,2,"[[1.0295492938399844, 1.0342086498432215, 1.04..."
341,"[[-929.1958618164062, -927.534423828125, -925....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",100,1000,BWA01.c3d,TVC60,2,"[[1.0422836482846283, 1.0410616700045021, 1.03..."


In [7]:
# Further function description can be found in 'utils.py': 
# This function is used to split a given dataset into train, test, and validation sets. 
# It splits the data based on the labels present in the dataset and assigns a certain proportion of data to each set.
# Stratifies patients based on Trial ID. All trials from each patient are either in the training, validation or test split.
    # train_data: This DataFrame contains the training data.
    # test_data: This DataFrame contains the test data.
    # train_validation_data: This DataFrame contains the training data for the validation set.
    # test_validation_data: This DataFrame contains the test data for the validation set.

hold_out = 0.3
train_data, test_data, train_validation_data, test_validation_data = get_train_test_split(dataset, hold_out)

In [8]:
# Mmaps specific gait phases to binary labels (0 or 1). It then adds random padding (5-125 frames) 
# around the first and last detected events to create robust training samples.
# 0 = no Event; 1 = event.

train_grf, train_velocity = event_seperator(copy.deepcopy(train_validation_data['GRF_Events']), copy.deepcopy(train_validation_data['Velocity']), is_IC)
validation_grf, validation_velocity = event_seperator(copy.deepcopy(test_validation_data['GRF_Events']), copy.deepcopy(test_validation_data['Velocity']), is_IC)

In [9]:
# Further function description in 'utils.py':
# This function reshapes and pads sequences of trajectory and GRF data to the longest input sequence,
# so that they can be used as input for the neural network. The input shape of train_velocity is (features, num_frames).
# The input shape of train_grf is (num_frames).
# Specifically, it pads the sequences with zeros so that all sequences have the same length 
# (the length of the longest sequence), and it reshapes the data to have the shape 
# (num_samples, max_seq_length, num_features) for train_velocity and (num_samples, num_frames) for train_grf, 
# as this is the input format for IntellEvent.  

train_velocity, train_grf = reshape_data(train_velocity, train_grf)
validation_velocity, validation_grf = reshape_data(validation_velocity, validation_grf)

In [10]:
# samples are HIGHLY imbalanced between no-events and events, therefore we use sample weights to 
# give more weight to rarely-seen events. A ratio of 1:10 (0.1 to 1) was found to be the best ratio.
sample_weights = get_sample_weights(train_grf, [0.1, 1])
train_grf = tf.expand_dims(train_grf.astype(np.float32), axis=-1)
validation_grf = tf.expand_dims(validation_grf.astype(np.float32), axis=-1)

2026-04-08 11:08:57.072354: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 9.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
2026-04-08 11:08:57.096976: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 9.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
2026-04-08 11:08:57.272389: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 18137 MB memory:  -> device: 0, name: NVIDIA H100 PCIe MIG 2g.20gb, pci bus id: 0000:82:00.0, compute capability: 9.0


In [11]:
test_validation_velocity, test_validation_grf = reshape_data(test_validation_data['Velocity'], test_validation_data['GRF_Events'])
val_labels = test_validation_data['Label']

In [ ]:
##########################################################################################################################################

#Class: 'intellevent_model' (HyperModel)
# Purpose:
    # Defines a search space for a Deep Learning model to identify the optimal architecture and 
    # training parameters for gait event detection using KerasTuner.

# Hyperparameters Tuned:
    # rnn_units: Integer (128 to 256) defining the capacity of the LSTM layers.
    # dense_units: Integer (32 to 192) defining the size of the TimeDistributed dense layer.
    # dropout: Float (0.0 to 0.3) to control regularization and prevent overfitting.
    # learning_rate: Categorical ([1e-4, 5e-4, 1e-3]) to optimize the Adam optimizer convergence.
    # gamma/alpha: Focal Loss parameters to handle class imbalance (focusing on hard-to-predict events).
    # batch_size: Categorical ([8, 16, 32]) defining the number of samples processed before the model is updated.

# Model Architecture:
    # 1. Input: Accepts sequences of shape (None, last_dim), supporting variable trial lengths.
    # 2. Masking: Skips computation for padded time steps (mask_value=0.).
    # 3. Recurrent Layers: Three stacked Bidirectional LSTM layers for processing temporal dependencies 
    #    in both forward and backward directions.
    # 4. Dense Head: A TimeDistributed ReLU dense layer followed by a Sigmoid output layer 
    #    providing a per-frame event probability.

# Description:
    # The build() method constructs the computational graph and compiles it with a Binary Focal Crossentropy 
    # loss function. The fit() method is overridden to allow KerasTuner to experiment with different 
    # batch sizes during the search trials. This ensures the best combination of architecture and 
    # optimization settings is discovered for the specific gait dataset.
##########################################################################################################################################

from tensorflow.keras import layers
from keras import backend as backend
from keras.callbacks import EarlyStopping
import keras_tuner as kt
from keras_tuner.engine.hyperparameters import HyperParameter
from tensorflow.keras.regularizers import l1


class intellevent_model(kt.HyperModel):
   
    def build(self, hp):
        backend.clear_session()
        deterministic_ops(21)
        
        rnn_input = keras.Input(shape=(None, last_dim))
        rnn_net = tf.keras.layers.Masking(mask_value=0.)(rnn_input)
        rnn_units = hp.Int("rnn_units", min_value=128, max_value=256, step=64)
        
        for rnn_layers in range(0,3):
            rnn_net = layers.Bidirectional(layers.LSTM(rnn_units, return_sequences=True))(rnn_net)

        units = hp.Int("dense_units", min_value=32, max_value=192, step=32)
        rnn_net = layers.Dropout(hp.Choice("dropout",  [0.0, 0.1, 0.2, 0.3]))(rnn_net)
        rnn_net = layers.TimeDistributed(layers.Dense(units, activation="relu"))(rnn_net)

             
        rnn_output = layers.TimeDistributed(layers.Dense(1, activation="sigmoid"))(rnn_net)
        model = tf.keras.Model(inputs=rnn_input, outputs=rnn_output)

        
        learning_rate = hp.Choice("learning_rate", [1e-4, 5e-4, 1e-3])        

        optimizer = tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        )

        model.compile(
             loss=tf.keras.losses.BinaryFocalCrossentropy(gamma=hp.Choice("gamma", [1.5, 2.0, 2.5]), alpha=hp.Choice("alpha", [0.2, 0.4, 0.6, 0.8])), 
             optimizer=optimizer,
            )
        
        return model

    def fit(self, hp, model, *args, **kwargs):
        
        return model.fit(
            *args,
            batch_size=hp.Choice("batch_size", [8,16,32]), 
            **kwargs,
        )


In [ ]:
# Purpose:
    # Configures the search strategy for finding the optimal model configuration. 
    # RandomSearch picks combinations of hyperparameters at random from the defined search space.

# Parameters:
    # objective: Sets the performance target to minimize 'val_mae', which is the error in gait event timing.
    # max_trials: Limits the number of unique model architectures to test (currently set to 1 for testing).
    # seed: Ensures the random selection of parameters is consistent across different environments.
    # directory/project_name: Defines the file system path for saving trial logs and the best-performing model weights.

# Description:
    # This block serves as the orchestrator for the hyperparameter optimization phase. 
    # It creates a persistent environment where different versions of the IntellEvent model 
    # can be trained, compared, and stored.
##########################################################################################################################################

tuner = kt.RandomSearch(
    intellevent_model(),
    objective=kt.Objective("val_mae", direction="min"),
    directory='search_results',
    project_name='intellevent_tuning',
    seed=21,
    hyperparameters=None,
    tune_new_entries=True,
    allow_new_entries=True,
    max_trials = 100
)

In [ ]:
# Purpose:
    # Executes the hyperparameter search process, training multiple model variations to identify 
    # the optimal configuration based on validation performance.

# Components:
    # EarlyStopping (es): 
        # Monitors 'val_mae' and halts training if no improvement is seen for 15 consecutive epochs. 
        # It restores the best weights found during training and begins monitoring only after epoch 10.
    # CheckMAE (mae_performance): 
        # A custom callback that evaluates and prints the Mean Absolute Error per pathology class 
        # at the end of every epoch.
    # tuner.search: 
        # The core execution command that passes training data, validation sets, and sample weights 
        # (to handle class imbalance) to the KerasTuner engine.

# Description:
    # This block initiates the automated trial process. For each trial, the tuner builds a model, 
    # trains it using the specified callbacks, and evaluates it against the validation data. 
    # The combination of EarlyStopping and CheckMAE ensures that training is efficient while 
    # providing granular insight into how the model performs across different gait pathologies.
##########################################################################################################################################

es = EarlyStopping(
    monitor='val_mae', 
    patience=15, 
    restore_best_weights=True, 
    verbose=1,
    mode="min",
    start_from_epoch=10
)

mae_performance = CheckMAE(test_validation_velocity, test_validation_grf, val_labels )


# start search
tuner.search( 
    train_velocity,
    train_grf,
    validation_data=(validation_velocity, validation_grf),
    callbacks=[mae_performance, es],
    sample_weight=sample_weights,
    shuffle=True,
    epochs=10)

In [ ]:
# Purpose:
    # Extracts the top-performing configuration from the completed search trials and initializes 
    # the final model architecture for production training.

# Components:
    # tuner.get_best_hyperparameters: 
        # Queries the tuner's database to retrieve the set of parameters that achieved the lowest 
        # 'val_mae' during the search phase.
    # tuner.hypermodel.build(best_hps): 
        # Re-invokes the 'build' function of the intellevent_model class using the optimal 
        # settings (e.g., best rnn_units, learning_rate, and dropout) to instantiate a fresh 
        # model object ready for the final training run.

# Description:
    # This block transitions from the experimentation phase to the final model creation. 
    # By programmatically retrieving the best values, it ensures that the model used for 
    # final evaluation on the test set is perfectly aligned with the results discovered 
    # during hyperparameter tuning.
##########################################################################################################################################

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"Optimal Hyperparameters: {best_hps.values}")

# Train final model
final_model = tuner.hypermodel.build(best_hps)

In [ ]:
# Train the best model again

es = EarlyStopping(
    monitor='val_mae', 
    patience=15, 
    restore_best_weights=True, 
    verbose=1,
    mode="min"
    )

mae_performance = CheckMAE(test_validation_velocity, test_validation_grf, val_labels, is_IC)

history = final_model.fit( 
    train_velocity,
    train_grf,
    validation_data=(validation_velocity, validation_grf),
    callbacks=[mae_performance, es],
    shuffle=True,
    epochs=100,
    batch_size=8,
    sample_weight=sample_weights,
)

In [ ]:
# Purpose:
    # Prepares the unseen test dataset for final evaluation by restructuring temporal sequences 
    # and generating model inferences.

# Components:
    # reshape_data: 
        # Processes the 'Velocity' features and 'GRF_Events' labels into a 3D tensor format 
        # (Samples, Time Steps, Features) compatible with the model's input layer.
    # Metadata Extraction (labels, ids, trials): 
        # Preserves original subject identifiers and pathology labels. This allows the 
        # subsequent evaluation functions to group results by pathology and identify 
        # specific trials where the model may have failed.
    # final_model.predict: 
        # Generates the probabilistic output for the test set. A batch size of 100 is 
        # used to optimize memory efficiency during the forward pass.

# Description:
    # This block represents the transition from model training to final performance 
    # assessment. By extracting subject IDs and trial names alongside the predictions, 
    # the pipeline maintains full traceability, enabling a detailed clinical analysis 
    # of the model's timing accuracy across different patient groups.
##########################################################################################################################################

test_velocity, test_grf = reshape_data(test_data['Velocity'], test_data['GRF_Events'])

labels = test_data['Label']
ids = test_data['DBid']
trials = test_data['Trial']

test_predictions = final_model.predict(test_velocity, batch_size=100)

In [ ]:
# Final evaluation on pathology (label) level
# more details in utils.py
ic_mae_all, ic_list, tp, fn = eval_te_data(labels, ids, trials, test_predictions, test_grf, True)

In [ ]:
# show MAE on patholog (Label) level (Results are in Frames - adapt to ms accordingly, depending on the capturing frequency): 
ic_mae_all

In [ ]:
# Save the results to a pickle file for further processing
# Create a DataFrame with group_choice as the pathology labels
# and the used model for easier plotting in seaborn

# Add which labels/Pathologies are present
group_choice = ["LABEL1", "LABEL2"]
df_results = pd.DataFrame()

for i in range(0,2):
    group_choices = np.full(len(ic_list[i]), group_choice[i]) 
    if i == 0:
        df_results = pd.DataFrame({"ic_list": np.array(ic_list[i]), "Pathology": group_choices, 
                                    "Model": "IntellEvent", "Label": "BL1", "TP": np.array(tp[i]), "FP": np.array(fp[i]), "FN": np.array(fn[i])})
    else:
        df_results = pd.concat([df_results, pd.DataFrame({"ic_list": np.array(ic_list[i]), "Pathology": group_choices, 
                                    "Model": "IntellEvent", "Label": "BL1", "TP": np.array(tp[i]), "FP": np.array(fp[i]), "FN": np.array(fn[i])})])
df_results.to_pickle("IntellEvent_IC_BL1.pkl")